# Clustering without labels, and how you would know

MichAl Academy, lesson 2.14.

Run each cell with **Shift+Enter**.

Everything in this track so far had labels. This notebook throws them away,
which is the situation most security data is actually in, and then uses them at
the end as an answer key.

That trick is the point. On real work there is no answer key, so the honest
question is not "did the clustering work" but **"what could I have known
without one?"** The answer turns out to be less than the textbook methods
suggest.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.datasets import load_digits, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                             adjusted_rand_score, normalized_mutual_info_score,
                             roc_auc_score, average_precision_score)

SEED = 0
digits = load_digits()
X = StandardScaler().fit_transform(digits.data)
y = digits.target
TRUE_K = len(set(y))
print(f"{len(y)} images, {X.shape[1]} features, {TRUE_K} true classes")
print("the labels are loaded but not used until section 3")


## 1. Choosing k, with the three rules you will be told to use

k-means needs the number of clusters up front. The three standard ways of
choosing it:

- **The elbow.** Plot inertia, the total squared distance from each point to
  its cluster centre, and look for a bend.
- **Silhouette.** How much closer each point is to its own cluster than to the
  next nearest. Higher is better.
- **Calinski-Harabasz.** The ratio of between-cluster to within-cluster
  spread. Higher is better.

The truth is ten. See whether any of them says so.


In [ ]:
print(f"{'k':>3} {'inertia':>10} {'drop':>8} {'silhouette':>11} {'Calinski-Harabasz':>18}")
prev = None
best_sil, best_ch = (0, -1.0), (0, -1.0)
for k in range(2, 21):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    sil = silhouette_score(X, km.labels_)
    ch = calinski_harabasz_score(X, km.labels_)
    if sil > best_sil[1]:
        best_sil = (k, sil)
    if ch > best_ch[1]:
        best_ch = (k, ch)
    drop = "" if prev is None else f"{prev - km.inertia_:.0f}"
    print(f"{k:>3} {km.inertia_:>10.0f} {drop:>8} {sil:>11.4f} {ch:>18.1f}")
    prev = km.inertia_

print(f"\nbest silhouette:        k={best_sil[0]}  ({best_sil[1]:.4f})")
print(f"best Calinski-Harabasz: k={best_ch[0]}  ({best_ch[1]:.1f})")
print(f"the truth:              k={TRUE_K}")


Silhouette says **12**. Calinski-Harabasz says **2**. The truth is **10**.

And look for the elbow in the drop column. It does not exist: the drops run
4726, 4014, 3404, then back **up** to 3612, then 2902, up again to 3153. There
is no bend to find, only a slow decline with noise on it.

Note also how small the silhouettes are. The best is 0.1580. Lesson 2.13
measured pure Gaussian noise reaching **0.35** in a two-dimensional embedding,
and this is real data with ten genuine classes in it scoring less than half
that. Silhouette is not comparable across spaces, which is precisely why it
cannot be read as "how real are these clusters".

**None of the three rules found the answer.** They are not useless, they are
weak, and the difference matters: use them to rule out obviously wrong values
of k, not to pick one.


## 2. DBSCAN does not ask for k, and that is not the relief it sounds like

DBSCAN grows clusters wherever points are dense enough, decides the number
itself, and can refuse to assign a point at all. What it needs instead is
`eps`, the radius it treats as "nearby".

The sensible way to choose eps is from the distribution of each point's
distance to its fifth-nearest neighbour, so sweep percentiles of that.


In [ ]:
MIN_SAMPLES = 5
nn = NearestNeighbors(n_neighbors=MIN_SAMPLES + 1).fit(X)
kdist = nn.kneighbors(X)[0][:, MIN_SAMPLES]      # column 0 is the point itself
print(f"distance to the 5th neighbour: median {np.median(kdist):.2f},"
      f" 10th pct {np.percentile(kdist, 10):.2f},"
      f" 90th pct {np.percentile(kdist, 90):.2f}")

print(f"\n{'percentile':>11} {'eps':>7} {'clusters':>9} {'unassigned':>11}")
for pct in (10, 25, 40, 50, 60, 75, 90):
    eps = float(np.percentile(kdist, pct))
    lab = DBSCAN(eps=eps, min_samples=MIN_SAMPLES).fit_predict(X)
    print(f"{pct:>11} {eps:>7.2f} {len(set(lab) - {-1}):>9}"
          f" {int((lab == -1).sum()):>11}")


The cluster count runs from **26 down to 2** across that sweep, and the number
of points DBSCAN declines to assign runs from 1,394 to 80 out of 1,797.

So the claim that DBSCAN frees you from choosing k is true and misleading. You
choose eps, and **eps chooses k**, less directly and less visibly.

One methodological note, because getting this wrong cost me an hour. My first
attempt swept eps over fractions of the dataset's diameter instead of the
neighbour distribution, which in 64 dimensions means values around 9 when the
useful range is 3 to 6. Every setting collapsed to one cluster and I nearly
wrote that DBSCAN fails on this data. It does not. **Tuning a method badly and
reporting the result is its own failure mode**, and it is easier to fall into
than the faults in lesson 2.12.


## 3. Now open the answer key

Adjusted Rand index compares a clustering against the true labels: 0 is chance,
1 is perfect. You would not have this on real work.


In [ ]:
print(f"{'method':<32} {'clusters':>9} {'ARI':>8} {'NMI':>8}")
methods = [
    ("k-means, k=10", KMeans(n_clusters=10, n_init=10, random_state=SEED).fit_predict(X)),
    ("k-means, k=12 (silhouette's pick)",
     KMeans(n_clusters=12, n_init=10, random_state=SEED).fit_predict(X)),
    ("k-means, k=17", KMeans(n_clusters=17, n_init=10, random_state=SEED).fit_predict(X)),
    ("k-means, k=2 (Calinski's pick)",
     KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(X)),
    ("Gaussian mixture, 10 diagonal",
     GaussianMixture(n_components=10, random_state=SEED,
                     covariance_type="diag").fit_predict(X)),
    ("agglomerative, 10, Ward", AgglomerativeClustering(n_clusters=10).fit_predict(X)),
    ("DBSCAN, eps at the 45th pct",
     DBSCAN(eps=float(np.percentile(kdist, 45)), min_samples=MIN_SAMPLES).fit_predict(X)),
]
for name, lab in methods:
    nc = len(set(lab) - {-1})
    print(f"{name:<32} {nc:>9} {adjusted_rand_score(y, lab):>8.4f}"
          f" {normalized_mutual_info_score(y, lab):>8.4f}")


Three things in that table.

**Agglomerative clustering with Ward linkage wins**, 0.6643 against k-means'
0.4679 at the same k. Nothing in section 1 would have told you to try it, and
nothing in section 1 would have told you it won.

**k-means at k=17 scores 0.6626**, which is better than k-means at the true
k=10. That is not a mistake. Splitting one digit's images into two clusters
costs the index far less than merging two digits into one does, so overshooting
k is the safer error. **The true number of classes is not the best number of
clusters**, and if you were tuning k against an answer key you would not choose
ten.

Which partly rescues the silhouette. Its pick of k=12 scored **0.5469**,
better than k=10's 0.4679. So it named the wrong number and produced the better
clustering, for the reason above. Calinski-Harabasz gets no such rescue: its
pick of k=2 scores **0.1368**, and the rule that was most confident was most
wrong.


## 4. Anomaly detection, and what a label is worth

Same question from the other side. Instead of grouping everything, find the few
rows that do not belong.

Set it up as a rare-event problem: every benign case from the breast cancer
data plus one in twenty of the malignant ones. Then compare four unsupervised
detectors, which never see a label, against a supervised forest that does.


In [ ]:
bc = load_breast_cancer()
Xb = StandardScaler().fit_transform(bc.data)
benign = np.flatnonzero(bc.target == 1)
malignant = np.flatnonzero(bc.target == 0)
rare = np.random.default_rng(SEED).choice(malignant, size=len(malignant) // 20,
                                          replace=False)
keep = np.concatenate([benign, rare])
Xa, ya = Xb[keep], (bc.target[keep] == 0).astype(int)    # 1 = anomaly
print(f"{len(ya)} rows, {ya.sum()} anomalies, base rate {ya.mean():.4f}")

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

# Score the unsupervised detectors out of fold too, so they are judged on rows
# they have not seen. Same footing as the supervised model apart from labels.
def out_of_fold_scores(make):
    s = np.zeros(len(ya))
    for tr, te in cv.split(Xa, ya):
        s[te] = -make().fit(Xa[tr]).score_samples(Xa[te])
    return s


print(f"\n{'':<38} {'ROC-AUC':>9} {'avg precision':>14}")
for name, make in (
    ("IsolationForest", lambda: IsolationForest(random_state=SEED, n_jobs=1)),
    ("OneClassSVM", lambda: OneClassSVM(gamma="scale", nu=0.05)),
    ("EllipticEnvelope", lambda: EllipticEnvelope(random_state=SEED, support_fraction=0.9)),
    ("PCA reconstruction error, 5 comps", None),
):
    if make is None:
        s = np.zeros(len(ya))
        for tr, te in cv.split(Xa, ya):
            p = PCA(n_components=5, random_state=SEED).fit(Xa[tr])
            s[te] = ((Xa[te] - p.inverse_transform(p.transform(Xa[te]))) ** 2).sum(axis=1)
    else:
        s = out_of_fold_scores(make)
    print(f"{name:<38} {roc_auc_score(ya, s):>9.4f} {average_precision_score(ya, s):>14.4f}")

# LocalOutlierFactor in its default mode scores the data it was fitted on, so
# it gets its own line rather than being forced into the loop.
lof = LocalOutlierFactor(novelty=False)
lof.fit(Xa)
s_lof = -lof.negative_outlier_factor_
print(f"{'LocalOutlierFactor (in sample)':<38} {roc_auc_score(ya, s_lof):>9.4f}"
      f" {average_precision_score(ya, s_lof):>14.4f}")

sup = cross_val_predict(RandomForestClassifier(random_state=SEED, n_jobs=1),
                        Xa, ya, cv=cv, method="predict_proba")[:, 1]
print(f"\n{'supervised forest, sees labels':<38} {roc_auc_score(ya, sup):>9.4f}"
      f" {average_precision_score(ya, sup):>14.4f}")
print(f"{'random':<38} {0.5:>9.4f} {ya.mean():>14.4f}")


Read the two columns against each other, because they disagree.

**By ROC-AUC the unsupervised detector matches the supervised one.** That would
mean labels are worth nothing here.

**By average precision it does not.** The supervised model is clearly ahead.

Which is lesson 2.7 arriving without being invited. The base rate is 0.0272, so
the negatives are 97% of the data, and a false positive rate divides by them.
Average precision divides by the alerts raised, which is the quantity a person
has to work through.


In [ ]:
# So ask the operational question instead. A team can investigate twenty cases.
print("if you can investigate 20 cases, how many of the 10 anomalies do you find?")
for name, s in (("IsolationForest, out of fold", out_of_fold_scores(
                    lambda: IsolationForest(random_state=SEED, n_jobs=1))),
                ("supervised forest", sup)):
    top = np.argsort(-s)[:20]
    print(f"  {name:<32} {int(ya[top].sum())} of {int(ya.sum())}")


Eight against nine.

That is the number worth carrying into a meeting about whether to fund
labelling. Not "supervised is better", which is true and unhelpful, but: **at a
budget of twenty investigations on this data, labels bought one extra detection
out of ten.** Whether that is worth the labelling effort is somebody's budget
decision, and now it is a decision with a number attached.

Run the same comparison on your own data before assuming either answer. The
gap will be different, and the direction of the argument depends on the gap.


## What to take from this

| Claim | What we measured |
|---|---|
| The elbow method finds the number of clusters | There was no elbow. The inertia drops were 4726, 4014, 3404, then back up to 3612 |
| Silhouette finds the number of clusters | It picked 12 where the truth is 10, though k=12 clustered better than k=10 did. Calinski-Harabasz picked 2, worth 0.1368 |
| A silhouette of 0.15 means weak clusters | It is real data with 10 real classes. Lesson 2.13's pure noise scored 0.35 |
| DBSCAN frees you from choosing k | It asks for eps instead, and eps moved the cluster count from 26 to 2 |
| The true class count is the best k | k-means at k=17 scored 0.6626 against 0.4679 at k=10 |
| k-means is the clustering default | Agglomerative Ward scored 0.6643 against k-means' 0.4679 on the same k |
| Unsupervised anomaly detection is nearly as good | By ROC-AUC yes, by average precision no, and at a 20-case budget it found 8 of 10 against 9 |

The habit: **when you have no labels, get some for a sample.** Two hundred
labelled rows will not train a model, and they will tell you whether your
unsupervised pipeline is finding anything, which is the question you cannot
otherwise answer.


## Try this

1. Rerun section 1 on `PCA(n_components=10)` output. On this data every method
   scored **worse** after reducing dimensions, which is the opposite of the
   usual advice. Check it, then check whether the same holds for your data.
2. Set `contamination=0.03` on `IsolationForest` to match the real base rate.
   The score ranking will not change, because contamination only moves where
   the cut falls, which is lesson 2.8's point again.
3. Take the anomaly comparison down to two labelled anomalies instead of ten
   for the supervised model. At what number of labels does the supervised
   advantage disappear? That is the real form of the funding question.
